# Prediksi kata berikutnya

Split dokumen sebelum membuat contoh prefix. Pengujian tidak boleh menyertakan kata jawaban di input.

Kode lengkap dari pipeline dan snippet asli. Ikuti urutan tahap dan ubah konfigurasi awal sesuai data. Alternatif tersedia pada cheatsheet utama.

## Setup & konfigurasi

Satu-satunya sel yang wajib diubah. Jalankan paling awal.

In [ ]:
import os
os.environ.update(HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1", TOKENIZERS_PARALLELISM="false")
from pathlib import Path
import importlib.metadata as md

def env_report(models_dir=None):
    import torch
    for p in ("torch", "transformers", "scikit-learn", "pandas", "numpy", "sentence-transformers", "lightgbm"):
        try:
            print(p, md.version(p))
        except md.PackageNotFoundError:
            print(p, "TIDAK ADA")
    gpu = torch.cuda.is_available()
    print("cuda", gpu, torch.cuda.get_device_name(0) if gpu else "")
    if gpu:
        print("VRAM GB", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "| bf16", torch.cuda.is_bf16_supported())
    if models_dir and Path(models_dir).exists():
        print("model lokal:", sorted(p.name for p in Path(models_dir).iterdir() if p.is_dir()))
    return gpu

import os
import random
import numpy as np
import torch

def seed_everything(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

# ===== jalankan =====
import re, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")
MODELS = Path(os.environ.get("COMP_ROOT", r"C:\gemastik")) / "models"
SEED = 42
seed_everything(SEED)
env_report(MODELS)

DATA, TEXT, K = "train.csv", "text", 5


## Muat data

Baca file, otomatis mencoba encoding, simpan parquet.

In [ ]:
from pathlib import Path
import pandas as pd

def read_any(path, cache=True, **kw):
    """df = read_any('train.csv'). Lain kali otomatis membaca train.parquet."""
    p = Path(path)
    if p.suffix == ".parquet":
        return pd.read_parquet(p, **kw)
    if cache and p.with_suffix(".parquet").exists():
        return pd.read_parquet(p.with_suffix(".parquet"))
    if p.suffix in (".jsonl", ".json"):
        return pd.read_json(p, lines=p.suffix == ".jsonl", **kw)
    if p.suffix in (".xlsx", ".xls"):
        return pd.read_excel(p, **kw)
    sep = kw.pop("sep", "\t" if p.suffix == ".tsv" else ",")
    for enc in ("utf-8", "utf-8-sig", "latin-1"):
        try:
            df = pd.read_csv(p, sep=sep, encoding=enc, on_bad_lines="warn", **kw)
            break
        except UnicodeDecodeError:
            continue
    if cache:
        df.to_parquet(p.with_suffix(".parquet"), index=False)
    return df

# ===== jalankan =====
df = read_any(DATA)
docs = df[TEXT].fillna("").astype(str).tolist()
print(len(docs))


## Validasi per dokumen

Bukan per kata: mencegah bocor.

In [ ]:
import numpy as np
from sklearn.model_selection import GroupShuffleSplit, train_test_split

def holdout(n_or_y, test_size=.2, groups=None, stratify=True, seed=42):
    """Kembalikan (idx_train, idx_valid)."""
    y = np.asarray(n_or_y)
    idx = np.arange(len(y))
    if groups is not None:
        return next(GroupShuffleSplit(1, test_size=test_size, random_state=seed).split(idx, groups=groups))
    return train_test_split(idx, test_size=test_size, random_state=seed, stratify=y if stratify else None)

# ===== jalankan =====
a, b = holdout(docs, stratify=False)


## N-gram

Backoff dari konteks terpanjang.

In [ ]:
import re
from collections import Counter, defaultdict

class NGramLM:
    def __init__(self, n=3):
        self.n, self.tab = n, [defaultdict(Counter) for _ in range(n)]
    def fit(self, docs):
        for d in docs:
            w = re.findall(r"\w+", str(d).lower())
            for i in range(len(w)):
                for o in range(self.n):
                    if i - o >= 0:
                        self.tab[o][tuple(w[i - o:i])][w[i]] += 1
        return self
    def topk(self, context, k=5):
        w = tuple(re.findall(r"\w+", str(context).lower()))
        out = []
        for o in range(self.n - 1, -1, -1):
            for x, _ in self.tab[o].get(w[-o:] if o else (), Counter()).most_common(k):
                if x not in out:
                    out.append(x)
            if len(out) >= k:
                break
        return out[:k]

# ===== jalankan =====
lm = NGramLM(3).fit([docs[i] for i in a])


## Top-k accuracy

Berapa kali kata benar ada di K tebakan.

In [ ]:
hit = n = 0
for i in b:
    w = re.findall(r"\w+", docs[i].lower())
    for j in range(2, len(w)):
        n += 1
        hit += w[j] in lm.topk(" ".join(w[:j]), K)
print("top-k accuracy", round(hit / max(n, 1), 4), "dari", n)
